# Day 1 Task: ATS Resume Gap Analyzer

This notebook builds a small **AI-powered resume assistant** that acts like an Applicant Tracking System (ATS) reviewer.

Given a job description, it calls the OpenAI API and returns actionable guidance on what a candidate should add to their resume to improve their match score (target: **70%+**).

## What you'll learn

- Loading API keys securely with `.env`
- Calling the OpenAI Responses API
- Structuring **system** and **user** prompts for a focused task
- Chaining helper functions into a simple pipeline

## Prerequisites

1. A `.env` file in this folder (or project root) with `OPENAI_API_KEY=your-key-here`
2. Dependencies installed: `openai`, `python-dotenv`
3. The correct Python kernel selected (your virtual environment)

## How to run

Execute cells **top to bottom** using `Shift + Enter`. Each section below explains what the following code does.

In [12]:
import os
from dotenv import load_dotenv
from openai import OpenAI

## 1. Environment Configuration

The next cells load your OpenAI API key from a `.env` file.

| Variable | Purpose |
|---|---|
| `OPENAI_API_KEY` | Authenticates requests to the OpenAI API |

`load_dotenv(override=True)` reads the `.env` file and sets environment variables. `override=True` ensures local values take precedence over any existing system variables.

> **Tip:** Never commit your `.env` file to git. Keep API keys private.

In [13]:
load_dotenv(override=True)
apiKey = os.getenv("OPENAI_API_KEY")

## 2. OpenAI Client

The `OpenAI()` client reads `OPENAI_API_KEY` from the environment automatically — no need to pass the key manually when creating the client.

In [14]:
client = OpenAI()

## 3. Prompt Design & Pipeline Functions

The cells below define the full request pipeline:

```
jobDescription
    → generateInput()       # builds system + user messages
    → getResponseFromAI()   # calls the LLM
    → getJobDescriptionKeyPoints()  # top-level entry point
```

### Function overview

| Function | Role |
|---|---|
| `callLLM(input)` | Sends a request to `gpt-4.1-mini` via the Responses API |
| `generateUserInput(jobDescription)` | Wraps the job description in a user prompt |
| `system_prompt` | Instructs the model to act as an ATS reviewer |
| `generateInput(jobDescription)` | Combines system + user messages into the API input format |
| `getResponseFromAI(input)` | Calls the LLM and extracts the text response |
| `getJobDescriptionKeyPoints(jobDescription)` | Orchestrates the full flow |

The system prompt tells the model to:
- Identify required skills from the job description
- Suggest what to add to a resume to pass a 70% ATS threshold
- Respond concisely in markdown

In [15]:
def callLLM(input): 
    return client.responses.create(
        model="gpt-4.1-mini",
        input=input
    )

In [16]:
def generateUserInput(jobDescription):
    return f"""
        Here is the job description. Scan and give me result.

        {jobDescription}
    """
    

In [17]:
system_prompt = """
    You are an ATS system that review candidates' resumes for job description. 
    You must help the candidate to tell what needs to be added in the resume based on the job description
    so that they can pass the threshold of 70%. 
    Share important information fromt the description like list of required skills. 
    Talk to the point. Response in markdown.
"""

In [18]:
def generateInput(jobDescription):
    return [
        { "role": "system", "content": system_prompt},
        { "role": "user", "content": generateUserInput(jobDescription)}
    ]

In [19]:
def getResponseFromAI(input):
    response = callLLM(input)
    return response.output_text

In [20]:
def getJobDescriptionKeyPoints(jobDescription): 
    input = generateInput(jobDescription)
    return getResponseFromAI(input)

## 4. Sample Job Description

The next cell contains a real-world job posting for a **Senior Software Engineer** role at Priority Technology Holdings.

This is the input the pipeline will analyze. You can replace it with any job description to test the assistant with different roles.

In [21]:
jobDescription = """
About the job
Job title: Senior Software Engineer

Reports to: Director, Engineering

Department: Development

Location: Waterloo, Ontario, Canada





About Priority: 

Priority Technology Holdings, Inc. is a leading financial technology company on a mission to deliver a personalized, easy-to-adopt financial toolset that accelerates cash flow and optimizes working capital for businesses. Our vision is to eliminate the barriers to unlocking revenue - empowering businesses to grow faster and operate smarter.



We achieve this through the Priority Commerce Engine, an innovative platform that combines payables, acquiring, and banking and treasury solutions. This unified approach allows businesses to streamline financial operations, reduce unnecessary costs, and uncover new revenue opportunities.



At Priority, we're driven by results. We expect our people to be known for results - bringing expertise, momentum, and relentless focus to every challenge, helping our clients and each other thrive.





About the Role:

This role is focused on designing, building, and scaling high-impact solutions that improve how we develop, operate, and optimize our platform. The Senior Software Engineer will lead technical design and implementation for complex initiatives, translating ambiguous problems into scalable technical solutions and helping move ideas from concept to production. This role requires strong technical judgment, architectural thinking, and the ability to balance rapid experimentation with production rigor, reliability, and maintainability.



This role is focused on designing, building, and scaling high-impact solutions across three core areas: customer-facing AI innovation, AI-driven risk and fraud, and AI-enabled operational efficiency.



You will lead the development of AI-powered product capabilities, helping prototype and productionize features that deliver meaningful value to customers. This includes working through ambiguity to translate new ideas into reliable, production-ready systems. You will also contribute to building and improving AI-driven risk and fraud systems, ensuring solutions are accurate, scalable, and responsive to evolving threats. In parallel, you will help develop AI-enabled internal tools and workflows that improve how we build and operate the platform, including developer experience, product development processes, and broader business functions.





Responsibilities: 

Owns the design, delivery, and operation of complex services or systems, ensuring high standards for reliability, scalability, and maintainability.
Leads technical design for features and systems, making sound architectural decisions and evaluating trade-offs across performance, reliability, and development velocity.
Drives implementation of solutions that align with system architecture, product requirements, and long-term platform evolution.
Designs and maintains comprehensive automated testing strategies that validate system behavior, prevent regressions, and support safe, rapid delivery.
Ensures systems integrate effectively into CI/CD pipelines with strong quality gates and reliable deployment practices.
Establishes and reinforces engineering standards through code reviews, design reviews, and mentorship.
Owns system reliability by defining and improving observability through metrics, logging, and tracing.
Designs and implements alerting strategies that detect issues early and align with system SLIs/SLOs.
Leads debugging and resolution of complex production issues, driving root cause analysis and long-term fixes.
Defines and tracks system-level KPIs (e.g., latency, error rates, throughput) and contributes to defining product KPIs that measure customer outcomes.
Ensures delivered solutions are measured, observable, and aligned with intended product and business impact.
Collaborates closely with software engineers, product managers, and product designers to shape solutions that deliver reliable and intuitive customer experiences.
Translates product requirements and user workflows into scalable technical designs that maintain system integrity and performance.
Identifies and addresses technical risks, scalability challenges, and reliability gaps across systems.
Drives improvements to system design, development workflows, and engineering practices within the team.
Leverages AI-assisted engineering tools to accelerate development, testing, debugging, and operational analysis while ensuring correctness and maintainability.






What Success Looks Like:

Owns systems that operate reliably, scale effectively, and meet defined performance and availability expectations.
Designs and delivers solutions that balance speed, quality, and long-term maintainability.
Technical decisions improve system architecture, reduce risk, and enable future product evolution.
Systems are observable, measurable, and supported by effective alerting and operational practices.
Production issues are identified, diagnosed, and resolved quickly, with durable fixes that reduce recurrence.
Testing strategies and validation systems provide strong confidence in system behavior and release quality.
System and product KPIs are clearly defined, measured, and used to evaluate success and guide improvements.
Engineering work is consistently aligned with customer outcomes and business impact.
Collaborates effectively with engineers, product managers, and product designers to deliver high-quality product experiences.
Provides technical leadership within the team and is a trusted partner in design and execution decisions.
Elevates team performance through mentorship, code reviews, and improved engineering practices.
Demonstrates strong ownership, accountability, and continuous improvement mindset.




Candidate Requirements:

Required:

5+ years of software engineering experience, with a track record of delivering high-quality software in production environments.
Proven ability to design, build, and maintain reliable services or systems with moderate complexity.
Strong experience contributing to technical design and making sound engineering decisions within a team.
Strong understanding of modern software development practices, including test-driven development (TDD), and building scalable, maintainable, and observable systems.
Experience designing and implementing APIs and services with clear, well-defined contracts.
Solid understanding of web application architecture, including RESTful APIs, backend services, and service-oriented design principles.
Strong understanding of data modeling and data access patterns, including writing efficient queries and contributing to well-structured relational schemas.
Experience working effectively within modern development environments, including version control systems (e.g., Git) and Agile development practices.
Ability to debug complex issues across services using logs, metrics, and traces.
Demonstrated ability to identify technical risks, evaluate trade-offs, and propose effective solutions.
Experience mentoring junior engineers and contributing to team engineering standards and practices.
Experience using AI-assisted development tools to improve productivity, testing, and debugging workflows.
Exposure to integrating AI-enabled features or workflows into applications is a plus.
The primary technology stack for the Innovation Studio team includes: Node.js/NestJS, AWS, Kubernetes, and modern CI/CD and observability tooling, along with AI platforms such as OpenAI, Anthropic, and Gemini. Candidates should have strong proficiency in backend and cloud-native development, with the ability to independently design and deliver complex solutions in distributed systems environments.


Preferred:

Experience in fintech, payments, lending, or other regulated financial systems.
Bachelor’s degree in Computer Science, Engineering, or a related technical field, or equivalent practical experience.
Advanced degree (e.g., Master’s) in a relevant technical discipline.




Work Environment & Culture:

We believe that performance and experience go hand in hand - an exceptional employee experience is earned through contribution. We are a results-driven team, grounded in our core values: ownership, authenticity, service, trust, innovation, and camaraderie.



Our culture is built for those who want to make an impact. We challenge each other to grow, celebrate progress, and support one another through shared goals and real connection. Whether you're building technology, serving clients, or supporting internal teams, you’ll be part of a company that empowers you to perform at your best and be known for results.





Compensation & Benefits:



Compensation range: $113k - $139k CAD

We invest in the whole employee - personally and professionally. Our benefits package is designed to support your well-being, growth, and success - both inside and outside of work.



Financial Wellness

Bonus programs
Financial wellness resources and employee discount programs


Health & Well-being

Medical, dental, and vision coverage
Mental health support for employees and dependents through Lyra Health
Family planning and women’s health benefits through Carrot
Gym membership reimbursement and virtual wellness programs (including yoga)


Time Off

3 weeks PTO to start, with unlimited PTO after year one


Growth & Development

Education expense reimbursement
Leadership development programs
Certified Payments Professional (CPP) certification support


We believe great performance starts with feeling supported - and we’ve built our benefits with that in mind.

 



Traditional Physical Requirements:

Requires prolonged sitting, standing, bending, stooping and stretching.
Requires the ability to lift 10 pounds.
Requires eye-hand coordination, manual dexterity and a normal range of hearing and vision (with or without correction).
 



Join our team at Priority Technology Holdings, Inc. and be part of a dynamic and innovative company that is transforming the financial technology landscape. Together, we can shape the future of payments and banking solutions while providing unmatched value to our clients.


Requirements added by the job poster

• 1+ years of work experience with Acceptance Test‚ÄìDriven Development (ATDD)

• 5+ years of work experience with RESTful WebServices

• 6+ years of work experience with Node.js
"""

## 5. Run the Analysis

The cell below calls `getJobDescriptionKeyPoints()` with the sample job description and prints the model's markdown response.

Expected output includes:
- A summary of **required skills and experience**
- **Preferred** qualifications
- Actionable suggestions for what to add to a resume

In [22]:
output = getJobDescriptionKeyPoints(jobDescription)
print(output)

```markdown
# Job Description Summary for Senior Software Engineer at Priority Technology Holdings, Inc.

## Required Skills & Experience:
- 5+ years in software engineering with production delivery.
- 6+ years experience with **Node.js**.
- 5+ years working with **RESTful Web Services**.
- 1+ year experience with **Acceptance Test–Driven Development (ATDD)**.
- Strong in designing, building, and maintaining reliable services/systems of moderate complexity.
- Architectural design skills with sound engineering decisions.
- Test-driven development (TDD) and automated testing strategies experience.
- Experience in API and service design with clear contracts.
- Understanding of web app architecture including RESTful APIs and service-oriented design.
- Strong data modeling and efficient querying skills.
- Familiarity with Git, Agile practices, CI/CD pipelines, metrics, logging, and tracing for observability.
- Debugging complex multi-service issues using logs, metrics, and traces.
- Mentori

## Next Steps

Try extending this notebook on your own:

1. **Swap the job description** — paste a posting you're actually applying to
2. **Add your resume** — pass both resume and job description to the model for a side-by-side gap analysis
3. **Tune the system prompt** — adjust the 70% threshold, tone, or output format
4. **Add error handling** — check that `OPENAI_API_KEY` is set before calling the API

---

*Built as part of the LLM Engineering course — Day 1 task.*